# 🎓 训练你自己的 AI 教师模型

> 从数据生成到模型部署 —— 完整的 LLM 微调实战

---

## 你将学到什么

| 章节 | 内容 | 核心技能 |
|------|------|----------|
| 第1章 | 为什么微调比 Prompt 工程更强 | 理解 fine-tuning 的价值 |
| 第2章 | QLoRA 原理可视化 | 矩阵分解、低秩适应 |
| 第3章 | 数据工厂 - 生成训练数据 | DataFactory 的使用 |
| 第4章 | 训练管线 - 一键运行 | 7 步训练管线 |
| 第5章 | 损失曲线分析 | loss 诊断与过拟合识别 |
| 第6章 | 模型评估 | BLEU/ROUGE/教育质量评分 |
| 第7章 | GGUF 转换与 Ollama 部署 | 模型格式转换 |
| 第8章 | 框架集成 | VersionManager 版本管理 |
| 第9章 | A/B 对比 | 模型对比与雷达图 |
| 第10章 | 练习与扩展 | 5 个实战任务 |

> 💡 **核心理念**：微调不是“重新训练一个模型”，而是在已有大模型的基础上，用少量高质量数据教会它特定领域的知识和风格。QLoRA 让这个过程在消费级 GPU 上就能完成。

In [ ]:
# ==================== 环境初始化 ====================import sys, os# 将项目根目录加入 Python 路径PROJECT_ROOT = 'E:/学习LLM/lumilearn'sys.path.insert(0, PROJECT_ROOT)os.chdir(PROJECT_ROOT)print(f"项目根目录: {PROJECT_ROOT}")print(f"当前工作目录: {os.getcwd()}")print(f"Python 版本: {sys.version}")# 检查关键模块是否可导入modules_ok = Truefor mod_name in ['models.distil.data_factory', 'models.distil.trainer',                  'models.distil.evaluate', 'models.version_manager']:    try:        __import__(mod_name)        print(f"  ✅ {mod_name}")    except ImportError as e:        print(f"  ❌ {mod_name}: {e}")        modules_ok = Falseif not modules_ok:    print("\n⚠️ 部分模块导入失败，请检查项目路径是否正确")else:    print("\n✅ 所有核心模块导入成功！")

---

## 第 1 章：为什么微调比 Prompt 工程更强？

### 动机

Base Model（基座模型）是“通才”——它知道很多东西，但不够专业。

比如你问 `qwen2.5:7b`：“请用费曼五步法讲解勾股定理”，它可能给出一个还行但不完全符合费曼风格的答案。

**微调 (Fine-tuning) 解决的问题**：
- 🎯 **风格一致**：让模型始终用费曼五步法回答
- 📚 **领域深度**：教育场景下的专业知识密度
- 🎨 **角色塑造**：加入“龙老师小蜜蜂”的幽默风格
- ⚡ **推理效率**：不需要在 prompt 里塞长篇指令

### 对比：Base Model vs Fine-tuned Model

| 维度 | Base Model | Fine-tuned Model |
|------|-----------|------------------|
| Prompt 长度 | 需要 200+ 字系统指令 | 只需简短问题 |
| 风格一致性 | 不稳定 | 高度一致 |
| 领域知识 | 广泛但浅 | 教育领域深入 |
| 推理成本 | 每次都要传长 prompt | prompt 短，token 少 |

下面我们来直观感受一下：

In [ ]:
# ==================== 第1章：Base vs Fine-tuned 对比 ====================import jsonfrom IPython.display import display, Markdown# 模拟 base model 和 fine-tuned model 对同一问题的回答comparison = {    "question": "请讲解勾股定理",    "base_model": {        "model": "qwen2.5:7b (base)",        "response": """勾股定理是直角三角形中三条边的关系。公式为 a² + b² = c²，其中 c 是斜边。这个定理在数学和物理中有广泛应用。证明方法有很多，比如面积法。""",        "stats": {"字数": 72, "风格": "教科书式", "例子": 0, "引导": "无"}    },    "fine_tuned_model": {        "model": "lumilearn-teacher (fine-tuned)",        "response": """🐝 龙老师小蜜蜂来啦！今天我们来聊聊勾股定理——这可是数学界的“明星定理”哦！【现象引入】你有没有发现，建筑工人在盖房子时，会用一根绳子量出3米、4米、5米的三角形来确定墙角是不是直角？这就是勾股定理的应用！【核心概念】勾股定理：直角三角形中，两条直角边（a, b）的平方和等于斜边（c）的平方。即 a² + b² = c²。【思维模型】想象一下：直角边是两条“垫脚石”，斜边是“捷径”。虽然捷径短，但它的“平方”恰好等于两条垫脚石平方之和。【自主推导】想一想：如果一个直角三角形的两条直角边分别是 6 和 8，斜边是多少？试试看！【30秒总结】记住：直角三角形 → 两短边平方和 = 长边平方 → a² + b² = c² ✓""",        "stats": {"字数": 335, "风格": "费曼五步法+幽默", "例子": 2, "引导": "自主推导"}    }}print("=" * 60)print("📊 同一问题，两种回答对比")print("=" * 60)print(f"\n问题: {comparison['question']}")for key, label in [("base_model", "🔵 基座模型"), ("fine_tuned_model", "🟣 微调模型")]:    m = comparison[key]    print(f"\n{label}: {m['model']}")    print(f"  {'─' * 50}")    print(f"  {m['response'][:300]}...")    print(f"  📊 统计: {json.dumps(m["stats"], ensure_ascii=False)}")print("\n" + "=" * 60)print("💡 结论: 微调模型在风格一致性、教学结构、引导性上明显优于基座模型")print("=" * 60)

---

## 第 2 章：QLoRA 原理可视化

### 什么是 QLoRA？

QLoRA = **Q**uantized **Lo**w-**R**ank **A**daptation

**两个核心思想**：

1. **量化 (Quantization)**：把模型权重的精度从 16-bit 降到 4-bit，大幅减少显存占用
2. **低秩适应 (LoRA)**：不修改原模型的所有参数，而是训练两个小矩阵（A 和 B），用它们的乘积来“修正”原模型的输出

### 为什么 LoRA 有效？

核心洞察：模型微调时，权重变化矩阵 ΔW 是**低秩**的。

这意味着：
- 不需要更新全部 7B 参数
- 只需要更新两个小矩阵 A (d×r) 和 B (r×d)
- 其中 r 是秩（rank），通常 r=16，远小于 d=4096

**参数量对比**：
- 全量微调：7B 参数
- LoRA (r=16)：约 0.1B 参数（仅 1.4%！）

下面我们用 numpy 来可视化 LoRA 的矩阵分解过程：

In [ ]:
# ==================== 第2章：QLoRA 矩阵分解可视化 ====================import numpy as npimport matplotlib.pyplot as pltfrom IPython.display import display, Markdown# 设置中文字体plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']plt.rcParams['axes.unicode_minus'] = Falseprint("=" * 60)print("🧮 QLoRA 矩阵分解可视化")print("=" * 60)# 模拟权重矩阵的维度d = 64      # 原始权重矩阵维度 (实际模型中是 4096)r = 8       # LoRA 秩 (rank)，远小于 dprint(f"\n📐 维度设定:")print(f"   原始权重矩阵 W: {d} × {d} = {d*d} 个参数")print(f"   LoRA 矩阵 A:    {d} × {r} = {d*r} 个参数")print(f"   LoRA 矩阵 B:    {r} × {d} = {r*d} 个参数")print(f"   LoRA 总参数:    {d*r + r*d} = {2*d*r} 个参数")print(f"   压缩比:         {d*d}/{2*d*r} = {d/(2*r):.1f}x")# 生成模拟的权重变化矩阵 ΔWnp.random.seed(42)# 构造一个低秩的 ΔW（真实的微调场景中 ΔW 确实是低秩的）U = np.random.randn(d, r) * 0.1V = np.random.randn(r, d) * 0.1delta_W_true = U @ V  # d × d 的低秩矩阵# LoRA: 训练小矩阵 A 和 B 来近似 ΔWA = np.random.randn(d, r) * 0.1B = np.random.randn(r, d) * 0.1delta_W_lora = A @ B  # A(d×r) × B(r×d) = d×d# 可视化fig, axes = plt.subplots(1, 4, figsize=(18, 4))# 1. 原始权重 W (模拟)W = np.random.randn(d, d) * 0.5im0 = axes[0].imshow(W, cmap='RdBu_r', aspect='auto')axes[0].set_title(f'原始权重 W\n({d}×{d} = {d*d} 参数)', fontsize=11)plt.colorbar(im0, ax=axes[0], shrink=0.8)# 2. LoRA 矩阵 Aim1 = axes[1].imshow(A, cmap='Blues', aspect='auto')axes[1].set_title(f'LoRA 矩阵 A\n({d}×{r} = {d*r} 参数)', fontsize=11)plt.colorbar(im1, ax=axes[1], shrink=0.8)# 3. LoRA 矩阵 Bim2 = axes[2].imshow(B, cmap='Reds', aspect='auto')axes[2].set_title(f'LoRA 矩阵 B\n({r}×{d} = {r*d} 参数)', fontsize=11)plt.colorbar(im2, ax=axes[2], shrink=0.8)# 4. 乘积 A×B (近似 ΔW)im3 = axes[3].imshow(delta_W_lora, cmap='RdBu_r', aspect='auto')axes[3].set_title(f'A×B ≈ ΔW\n({d}×{d}, 秩={r})', fontsize=11)plt.colorbar(im3, ax=axes[3], shrink=0.8)plt.suptitle('QLoRA 核心思想：用小矩阵的乘积近似权重变化', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()# 计算近似误差frobenius_error = np.linalg.norm(delta_W_true - delta_W_lora, 'fro')print(f"\n📊 Frobenius 范数误差: {frobenius_error:.4f}")print(f"\n💡 核心洞察:")print(f"   - 原始 W 有 {d*d} 个参数需要更新")print(f"   - LoRA 只需要训练 {2*d*r} 个参数（A 和 B）")print(f"   - 节省了 {(1 - 2*d*r/(d*d))*100:.1f}% 的参数量！")print(f"   - 实际 Qwen2.5-7B 中: d=4096, r=16 → 节省 99.2% 参数")

In [ ]:
# ==================== 第2章：秩 (rank) 对近似质量的影响 ====================print("\n" + "=" * 60)print("📈 秩 (r) 对近似质量的影响")print("=" * 60)d_large = 128ranks = [1, 2, 4, 8, 16, 32, 64]errors = []np.random.seed(123)# 构造一个真正低秩的 ΔW（秩=8）U_true = np.random.randn(d_large, 8) * 0.1V_true = np.random.randn(8, d_large) * 0.1delta_true = U_true @ V_truefor r_val in ranks:    A_r = np.random.randn(d_large, r_val) * 0.1    B_r = np.random.randn(r_val, d_large) * 0.1    delta_approx = A_r @ B_r    err = np.linalg.norm(delta_true - delta_approx, 'fro')    errors.append(err)    print(f"  r={r_val:2d}: 误差 = {err:.4f}, 参数量 = {2*d_large*r_val} ({2*d_large*r_val/(d_large*d_large)*100:.1f}%)")# 绘图fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))ax1.plot(ranks, errors, 'o-', color='#667eea', linewidth=2, markersize=8)ax1.set_xlabel('LoRA Rank (r)', fontsize=12)ax1.set_ylabel('Frobenius 误差', fontsize=12)ax1.set_title('秩越大 → 近似误差越小', fontsize=13, fontweight='bold')ax1.grid(True, alpha=0.3)ax1.axvline(x=8, color='red', linestyle='--', alpha=0.5, label='真实秩=8')ax1.legend()param_pct = [2 * d_large * r_val / (d_large * d_large) * 100 for r_val in ranks]ax2.bar(range(len(ranks)), param_pct, color=['#667eea' if r_val <= 16 else '#f5576c' for r_val in ranks])ax2.set_xticks(range(len(ranks)))ax2.set_xticklabels([f'r={r_val}' for r_val in ranks])ax2.set_ylabel('参数量占比 (%)', fontsize=12)ax2.set_title('LoRA 参数量 vs 原始模型', fontsize=13, fontweight='bold')ax2.axhline(y=100, color='red', linestyle='--', alpha=0.5, label='全量微调=100%')ax2.legend()plt.suptitle('LoRA Rank 选择：r=16 是常见的最佳折中', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()print("\n💡 实践建议:")print("   - r=4~8:  快速实验，适合小数据集")print("   - r=16:   标准配置，LumiLearn 默认值")print("   - r=32~64: 更高质量，需要更多显存")print("   - r > 64:  收益递减，不如增大数据量")

---

## 第 3 章：数据工厂 —— 生成你的第一批训练数据

### 训练数据格式

LumiLearn 使用 **instruction-response 对** 格式（JSONL）：

```json
{"instruction": "请讲解勾股定理...", "response": "勾股定理是...", "subject": "math", "topic": "勾股定理"}
```

### 费曼五步法结构

每条训练数据都遵循费曼五步教学法：
1. **现象引入** - 日常场景引出问题
2. **认知冲突** - 激发好奇心
3. **思维模型** - 用类比/比喻解释
4. **自主推导** - 引导自己推导
5. **30秒测试** - 一句话总结

下面我们用 `DataFactory` 来预览模板（dry-run 模式，不调用 API）：

In [ ]:
# ==================== 第3章：DataFactory 预览模板 ====================from models.distil.data_factory import DataFactoryfrom models.distil.prompts import get_subject_training_templates, SUBJECT_TRAINING_TEMPLATESprint("=" * 60)print("📋 可用学科模板总览")print("=" * 60)# 列出所有可用模板templates = get_subject_training_templates()subjects = set(t["subject"] for t in templates)for s in sorted(subjects):    st = get_subject_training_templates(s)    topics = [t["topic"] for t in st]    print(f"\n  📚 {s} ({len(st)} 个模板)")    for topic in topics:        print(f"      - {topic}")print(f"\n{'=' * 60}")print(f"📊 总计: {len(templates)} 个模板, {len(subjects)} 个学科")print(f"{'=' * 60}")

In [ ]:
# ==================== 第3章：Dry-run 预览具体模板 ====================print("\n" + "=" * 60)print("🧪 Dry-Run: 预览模板内容（不调用 API）")print("=" * 60)factory = DataFactory()# 为数学学科生成 2 条 dry-run 数据dry_run_data = factory.generate_subject_data("math", count=2, dry_run=True)print("\n" + "=" * 60)print("📝 生成的预览数据条目:")print("=" * 60)for i, entry in enumerate(dry_run_data):    print(f"\n  [{i+1}] 学科: {entry['subject']} | 主题: {entry['topic']}")    print(f"  Instruction: {entry['instruction'][:150]}...")    print(f"  Response: {entry['response'][:100]}...")print("\n💡 DRY-RUN 模式不调用 Ollama API，只展示模板结构")print("   去掉 dry_run=True 即可实际生成数据")

In [ ]:
# ==================== 第3章：实际生成训练数据（需 Ollama 运行） ====================print("\n" + "=" * 60)print("🔄 实际生成训练数据（需要 Ollama 运行）")print("=" * 60)try:    import requests    # 检查 Ollama 是否可用    resp = requests.get("http://192.168.2.xx:11434/api/tags", timeout=5)    if resp.status_code == 200:        models = resp.json().get("models", [])        model_names = [m["name"] for m in models]        print(f"✅ Ollama 可用，已加载模型: {model_names}")                # 选择可用的模型        target_model = None        for preferred in ["qwen2.5:7b", "qwen2.5:14b", "qwen2:7b"]:            if preferred in model_names:                target_model = preferred                break                if target_model:            print(f"\n📦 使用模型: {target_model}")            print("⏳ 生成数学学科 3 条训练数据...")                        factory_real = DataFactory(model=target_model)            real_data = factory_real.generate_subject_data("math", count=3, dry_run=False)                        if real_data:                # 保存到临时文件                output_path = "data/distil/demo_train_data.jsonl"                factory_real.save_to_jsonl(real_data, output_path)                                # 显示统计                factory_real.compute_statistics(output_path)                                # 显示第一条数据                print(f"\n📝 示例数据 (第1条):")                print(f"   Instruction: {real_data[0]['instruction'][:120]}...")                print(f"   Response: {real_data[0]['response'][:200]}...")            else:                print("❌ 数据生成失败")        else:            print("⚠️ 未找到 qwen2.5 模型，请先运行: ollama pull qwen2.5:7b")    else:        print(f"⚠️ Ollama 返回状态码: {resp.status_code}")except requests.exceptions.ConnectionError:    print("⚠️ 无法连接到 Ollama 服务 (http://192.168.2.xx:11434)")    print("   跳过实际数据生成，请确保 Ollama 正在运行")    print("\n   💡 手动生成数据命令:")    print("   python -m models.distil.data_factory --subject math --count 5 --output data/distil/train_data.jsonl")except Exception as e:    print(f"⚠️ 发生错误: {e}")    print("   跳过实际数据生成")

In [ ]:
# ==================== 第3章：数据统计与分布可视化 ====================import matplotlib.pyplot as pltimport jsonprint("\n" + "=" * 60)print("📊 训练数据分布可视化")print("=" * 60)# 模拟训练数据的学科分布（实际训练时会是真实数据）subject_dist = {"math": 100, "physics": 80, "chemistry": 60, "chinese": 70, "english": 90, "programming": 50}response_lengths = {    "math": [350, 420, 380, 500, 320, 450, 390, 410, 370, 480],    "physics": [400, 380, 450, 420, 390, 460, 410, 370],    "english": [300, 350, 280, 320, 340, 310, 290, 330, 360],}fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))# 学科分布饼图colors = ['#667eea', '#764ba2', '#f093fb', '#4facfe', '#43e97b', '#fa709a']subjects = list(subject_dist.keys())counts = list(subject_dist.values())wedges, texts, autotexts = ax1.pie(    counts, labels=subjects, autopct='%1.1f%%',    colors=colors[:len(subjects)], startangle=90)ax1.set_title('训练数据学科分布', fontsize=13, fontweight='bold')# 回复长度分布bp = ax2.boxplot(    [response_lengths[s] for s in response_lengths],    labels=list(response_lengths.keys()), patch_artist=True)for patch, color in zip(bp['boxes'], colors[:len(response_lengths)]):    patch.set_facecolor(color)    patch.set_alpha(0.6)ax2.set_ylabel('Response 长度 (字符)', fontsize=12)ax2.set_title('各学科回复长度分布', fontsize=13, fontweight='bold')ax2.grid(True, alpha=0.3, axis='y')plt.suptitle('训练数据质量概览', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()print("\n💡 数据质量要点:")print("   - 学科分布应尽量均衡，避免某个学科数据过多")print("   - 回复长度建议 300-800 字符，太短缺乏深度，太长浪费 token")print("   - 确保每条数据都遵循费曼五步法结构")

---

## 第 4 章：训练管线 —— 一键运行

### 7 步训练管线

LumiLearn 的 `train_lumilearn.sh` 封装了完整的训练流程：

```
Step 1: 环境检测       → Python/CUDA/Ollama/磁盘空间
Step 2: 数据准备       → DataFactory 生成 JSONL 训练数据
Step 3: QLoRA 微调     → 4-bit 量化 + LoRA 高效训练
Step 4: 权重合并+GGUF  → 合并 adapter + 转换为 GGUF 格式
Step 5: 创建 Ollama 模型 → 生成 Modelfile + ollama create
Step 6: 模型评估       → BLEU/ROUGE/教育质量评分
Step 7: 框架注册       → 更新 config/models.yaml
```

下面我们来看看每一步的具体操作：

In [ ]:
# ==================== 第4章：训练管线概览 ====================import subprocessimport platformprint("=" * 60)print("🔧 训练管线 7 步概览")print("=" * 60)steps = [    ("Step 1", "环境检测", "检查 Python3/pip3/Ollama/CUDA/磁盘空间/依赖包"),    ("Step 2", "数据准备", "DataFactory 生成 JSONL 训练数据 + 清洗去重"),    ("Step 3", "QLoRA 微调", "4-bit 量化 + LoRA (r=16) 高效微调"),    ("Step 4", "GGUF 转换", "合并 adapter 权重 + 转换 + 量化"),    ("Step 5", "Ollama 部署", "生成 Modelfile + ollama create"),    ("Step 6", "模型评估", "BLEU-4/ROUGE-L/教育质量/费曼度/延迟"),    ("Step 7", "框架注册", "更新 config/models.yaml + VersionManager"),]for step, name, desc in steps:    print(f"\n  {step} | {name}")    print(f"  {'─' * 50}")    print(f"  {desc}")print(f"\n{'=' * 60}")print("📋 完整命令示例:")print("  bash train_lumilearn.sh --model-name lumilearn-v1 --sample-count 100")print("  bash train_lumilearn.sh --cpu-only --sample-count 50")print("  bash train_lumilearn.sh --dry-run  # 预览模式")print(f"{'=' * 60}")

In [ ]:
# ==================== 第4章：Dry-run 训练管线 ====================print("\n" + "=" * 60)print("🧪 模拟 Dry-Run: 展示训练管线将执行的操作")print("=" * 60)import os# 检查 train_lumilearn.sh 是否存在script_path = os.path.join(PROJECT_ROOT, "train_lumilearn.sh")if os.path.exists(script_path):    print(f"✅ 训练脚本已找到: {script_path}")        # 在 Windows 上无法直接运行 bash，所以模拟输出    if platform.system() == "Windows":        print("\n⚠️ Windows 环境，无法直接运行 bash 脚本")        print("   以下是 dry-run 模式将展示的内容:")        print()                # 模拟 dry-run 输出        mock_output = """  [DRY-RUN] Step 1: 环境检测  [DRY-RUN] 检查 Python3/CUDA/Ollama...    [DRY-RUN] Step 2: 数据准备  [DRY-RUN] python -m models.distil.data_factory --subjects math,physics,chinese,english --count 100    [DRY-RUN] Step 3: QLoRA 微调训练  [DRY-RUN] python -m models.distil.trainer --model_name Qwen/Qwen2.5-7B-Instruct --data_path data/distil/train_data.jsonl    [DRY-RUN] Step 4: 权重合并 + GGUF 转换  [DRY-RUN] export_gguf(adapter_path, output_path, quantize='q4_k_m')    [DRY-RUN] Step 5: 创建 Ollama 模型  [DRY-RUN] ollama create lumilearn-custom-v1 -f Modelfile    [DRY-RUN] Step 6: 模型评估  [DRY-RUN] python -m models.distil.evaluate --model lumilearn-custom-v1    [DRY-RUN] Step 7: 框架注册  [DRY-RUN] 更新 config/models.yaml 添加 lumilearn-custom-v1  """        print(mock_output)    else:        # Linux/Mac: 尝试运行 dry-run        try:            result = subprocess.run(                ["bash", script_path, "--dry-run"],                capture_output=True, text=True, timeout=30,                cwd=PROJECT_ROOT            )            print(result.stdout)            if result.stderr:                print("STDERR:", result.stderr[:500])        except Exception as e:            print(f"⚠️ 运行失败: {e}")else:    print(f"❌ 训练脚本未找到: {script_path}")print("\n💡 使用方式:")print("   # 完整训练")print("   bash train_lumilearn.sh --model-name my-teacher-v1 --sample-count 200")print("   # CPU 模式")print("   bash train_lumilearn.sh --cpu-only --sample-count 50")print("   # 跳过某些步骤")print("   bash train_lumilearn.sh --skip-data --skip-train --skip-export")

In [ ]:
# ==================== 第4章：迷你训练示例（可选） ====================print("\n" + "=" * 60)print("🔬 迷你训练示例（可选，需要 GPU 和依赖）")print("=" * 60)print("\n⚠️ 迷你训练需要以下条件:")print("   1. CUDA GPU 或 CPU（会很慢）")print("   2. 已安装 transformers, peft, bitsandbytes, accelerate, datasets")print("   3. 至少 10GB 磁盘空间")# 检查依赖deps_ok = Truefor dep in ['torch', 'transformers', 'peft', 'bitsandbytes', 'accelerate', 'datasets']:    try:        __import__(dep)        print(f"  ✅ {dep}")    except ImportError:        print(f"  ❌ {dep} 未安装")        deps_ok = Falseif not deps_ok:    print("\n⚠️ 依赖不完整，跳过迷你训练")    print("   安装命令: pip install transformers peft bitsandbytes accelerate datasets torch")else:    print("\n✅ 所有依赖已安装")    print("\n💡 要运行迷你训练，请执行:")    print("   python -m models.distil.trainer --data_path data/distil/demo_train_data.jsonl --output_dir models/distil/demo_adapter --epochs 1 --cpu-only")    print("\n   (添加 --cpu-only 以避免 GPU 需求)")

---

## 第 5 章：损失曲线分析

### 损失曲线告诉我们什么？

训练损失 (loss) 是衡量模型“学得怎么样”的核心指标：

- **健康曲线**：平稳下降，最终收敛
- **过拟合**：训练 loss 持续下降，但验证 loss 上升
- **欠拟合**：loss 下降缓慢或停滞在高位
- **学习率过高**：loss 震荡剧烈
- **学习率过低**：loss 下降极慢

下面我们用合成数据来可视化这些典型场景：

In [ ]:
# ==================== 第5章：损失曲线可视化 ====================import numpy as npimport matplotlib.pyplot as pltprint("=" * 60)print("📈 损失曲线分析")print("=" * 60)np.random.seed(42)steps = np.arange(0, 500)# 1. 健康曲线：指数衰减 + 小噪声healthy_loss = 3.0 * np.exp(-steps / 150) + 0.5 * np.exp(-steps / 30) + 0.3healthy_loss += np.random.normal(0, 0.05, len(steps))# 2. 过拟合：训练 loss 持续下降，验证 loss 先降后升train_loss_of = 3.0 * np.exp(-steps / 150) + 0.5 * np.exp(-steps / 30) + 0.2train_loss_of += np.random.normal(0, 0.03, len(steps))val_loss_of = 3.0 * np.exp(-steps / 150) + 0.3 + 0.001 * stepsval_loss_of += np.random.normal(0, 0.08, len(steps))# 3. 欠拟合：loss 下降缓慢underfit_loss = 3.5 * np.exp(-steps / 500) + 1.5underfit_loss += np.random.normal(0, 0.06, len(steps))# 4. 学习率过高：震荡lr_high_loss = 2.5 * np.exp(-steps / 200) + 0.5lr_high_loss += np.random.normal(0, 0.3, len(steps))# 绘图fig, axes = plt.subplots(2, 2, figsize=(14, 10))# 健康曲线axes[0, 0].plot(steps, healthy_loss, color='#43e97b', linewidth=1.5, alpha=0.7)axes[0, 0].plot(steps, np.convolve(healthy_loss, np.ones(20)/20, mode='same'),               color='#2d8a4e', linewidth=2, label='平滑后')axes[0, 0].set_title('✅ 健康曲线：平稳下降收敛', fontsize=13, fontweight='bold')axes[0, 0].set_ylabel('Loss', fontsize=12)axes[0, 0].legend()axes[0, 0].grid(True, alpha=0.3)# 过拟合axes[0, 1].plot(steps, train_loss_of, color='#667eea', linewidth=1.5, alpha=0.7, label='训练 loss')axes[0, 1].plot(steps, val_loss_of, color='#f5576c', linewidth=1.5, alpha=0.7, label='验证 loss')axes[0, 1].axvline(x=250, color='red', linestyle='--', alpha=0.5, label='分叉点')axes[0, 1].set_title('⚠️ 过拟合：训练↓ 验证↑', fontsize=13, fontweight='bold')axes[0, 1].legend()axes[0, 1].grid(True, alpha=0.3)# 欠拟合axes[1, 0].plot(steps, underfit_loss, color='#f093fb', linewidth=1.5, alpha=0.7)axes[1, 0].axhline(y=1.5, color='red', linestyle='--', alpha=0.5, label='停滞线')axes[1, 0].set_title('❌ 欠拟合：下降缓慢，停滞高位', fontsize=13, fontweight='bold')axes[1, 0].set_xlabel('训练步数', fontsize=12)axes[1, 0].set_ylabel('Loss', fontsize=12)axes[1, 0].legend()axes[1, 0].grid(True, alpha=0.3)# 学习率过高axes[1, 1].plot(steps, lr_high_loss, color='#fa709a', linewidth=1, alpha=0.5)axes[1, 1].plot(steps, np.convolve(lr_high_loss, np.ones(20)/20, mode='same'),               color='#c0392b', linewidth=2, label='平滑后')axes[1, 1].set_title('⚡ 学习率过高：剧烈震荡', fontsize=13, fontweight='bold')axes[1, 1].set_xlabel('训练步数', fontsize=12)axes[1, 1].legend()axes[1, 1].grid(True, alpha=0.3)plt.suptitle('训练损失曲线 —— 四种典型场景', fontsize=16, fontweight='bold')plt.tight_layout()plt.show()print("\n📊 诊断总结:")print("   ✅ 健康:    loss 平滑下降 → 训练正常")print("   ⚠️ 过拟合:  训练↓↓ 验证↑  → 需要更多数据或正则化")print("   ❌ 欠拟合:  停滞高位       → 增大模型或增加训练")print("   ⚡ 震荡:    剧烈波动       → 降低学习率")

In [ ]:
# ==================== 第5章：从实际训练日志提取 loss ====================print("\n" + "=" * 60)print("📊 检查实际训练统计")print("=" * 60)import jsonimport os# 检查是否有训练统计文件stats_path = "models/distil/adapter/training_stats.json"if os.path.exists(stats_path):    with open(stats_path, "r", encoding="utf-8") as f:        stats = json.load(f)    print("✅ 找到训练统计文件:")    for key, value in stats.items():        print(f"   {key}: {value}")else:    print("⚠️ 未找到训练统计文件 (models/distil/adapter/training_stats.json)")    print("   训练完成后会自动生成此文件")    print()    print("   💡 模拟训练统计:")    mock_stats = {        "model_name": "Qwen/Qwen2.5-7B-Instruct",        "lora_r": 16, "lora_alpha": 32,        "learning_rate": 0.0002,        "batch_size": 4, "epochs": 3,        "train_samples": 450,        "train_loss": 0.8234,        "train_runtime": 3600.0,        "device": "GPU",        "finished_at": "2026-06-03 15:30:00"    }    for key, value in mock_stats.items():        print(f"   {key}: {value}")

---

## 第 6 章：模型评估 —— 你的模型有多好？

### 评估指标说明

LumiLearn 使用四类指标全面评估模型：

| 指标 | 全称 | 测量内容 | 范围 |
|------|------|----------|------|
| **BLEU-4** | Bilingual Evaluation Understudy | n-gram 精度（1-4 gram） | 0-1 |
| **ROUGE-L** | Recall-Oriented Understudy for Gisting | 最长公共子序列召回 | 0-1 |
| **教育质量** | 五维度评分 | 正确性/完整性/难度/引导/趣味 | 0-10 |
| **费曼度** | 费曼风格检测 | 五步法遵循程度 | 0-100% |

下面我们来看这些指标是如何计算的：

In [ ]:
# ==================== 第6章：BLEU/ROUGE 评分演示 ====================from models.distil.evaluate import compute_bleu, compute_rouge_l, tokenize, get_ngramsprint("=" * 60)print("📊 BLEU-4 & ROUGE-L 评分演示")print("=" * 60)# 测试数据reference = "勾股定理：直角三角形中，两条直角边的平方和等于斜边的平方，即 a² + b² = c²。"test_cases = [    ("好答案", "勾股定理指出，在直角三角形中，直角边a和b的平方和等于斜边c的平方，公式为a² + b² = c²。"),    ("中等答案", "勾股定理是一个关于直角三角形的定理，a²+b²=c²。"),    ("差答案", "勾股定理很重要。"),    ("完全无关", "今天天气真好。"),]print(f"\n参考答案: {reference}")print(f"\n{'─' * 60}")print(f"{'类型':<10} {'BLEU-4':<10} {'ROUGE-L F1':<12} {'ROUGE-L P':<12} {'ROUGE-L R':<12}")print(f"{'─' * 60}")for label, candidate in test_cases:    bleu = compute_bleu(reference, candidate)    rouge = compute_rouge_l(reference, candidate)    print(f"{label:<10} {bleu:<10.4f} {rouge['f1']:<12.4f} {rouge['precision']:<12.4f} {rouge['recall']:<12.4f}")print(f"{'─' * 60}")print("\n💡 解读:")print("   - BLEU 衡量 n-gram 重叠度，越高越接近参考答案")print("   - ROUGE-L 基于最长公共子序列，兼顾召回和精度")print("   - 好答案 BLEU/ROUGE 都高，差答案分数低")print("   - 教育场景中 BLEU 0.3+, ROUGE-L 0.5+ 即可接受")

In [ ]:
# ==================== 第6章：教育质量五维度评分 ====================from models.distil.evaluate import evaluate_education_quality, detect_feynman_styleprint("\n" + "=" * 60)print("🎓 教育质量五维度 + 费曼风格检测")print("=" * 60)# 模拟一个高质量的回答good_response = """🐝 龙老师来啦！今天我们来学习勾股定理！【现象引入】你有没有发现，工人叔叔在盖房子时，会用绳子量出3米、4米、5米来确定直角？【核心概念】勾股定理的核心就是：直角三角形中，两条直角边的平方和等于斜边的平方。公式：a² + b² = c²。是不是很简单？【思维模型】想象一下：直角边是两条“路”，斜边是“捷径”。虽然捷径短，但它的“平方”恰好等于两条路平方之和。就像两条小路合起来等于一条大路！【自主推导】想一想：如果直角边是 6 和 8，斜边是多少？试试看：6² + 8² = 36 + 64 = 100 = 10²，所以斜边是 10！【30秒总结】记住一句话：直角三角形 → 两短边平方和 = 长边平方 → a² + b² = c² ✓"""# 教育质量评分edu = evaluate_education_quality(good_response, "勾股定理")print("\n教育质量五维度评分:")dim_labels = {    "accuracy": "正确性", "completeness": "完整性",    "difficulty_adaptation": "难度适应", "guidance": "引导性",    "engagement": "趣味性"}for dim, label in dim_labels.items():    score = edu[dim]    bar = "█" * int(score) + "░" * (10 - int(score))    print(f"  {label:<8} [{bar}] {score:.1f}/10")# 费曼风格检测feynman = detect_feynman_style(good_response)print(f"\n费曼风格检测:")print(f"  费曼度: {feynman['score']:.1f}%")print(f"  检测到步骤: {feynman['detected_steps']}")print(f"  各步骤置信度:")for step, score in feynman['step_scores'].items():    bar = "█" * int(score * 10) + "░" * (10 - int(score * 10))    print(f"    {step:<12} [{bar}] {score:.3f}")print("\n💡 评分基于关键词启发式算法，实际使用中配合 Judge 模型更准确")

In [ ]:
# ==================== 第6章：运行模型评估（需 Ollama） ====================print("\n" + "=" * 60)print("🔬 运行模型评估（需要 Ollama 运行）")print("=" * 60)try:    import requests    from models.distil.evaluate import ModelEvaluator, DEFAULT_TEST_SET        # 检查 Ollama    resp = requests.get("http://192.168.2.xx:11434/api/tags", timeout=5)    if resp.status_code == 200:        models = [m["name"] for m in resp.json().get("models", [])]                # 找到可用的 qwen 模型        available = [m for m in models if "qwen" in m.lower()]        if available:            target = available[0]            print(f"✅ 使用模型: {target}")            print(f"   测试集大小: {len(DEFAULT_TEST_SET)} 题")            print("\n⏳ 正在评估（仅评估前2题作为演示）...")                        evaluator = ModelEvaluator()            # 只评估前2题以节省时间            mini_test = DEFAULT_TEST_SET[:2]            results = evaluator.evaluate_model(target, test_set=mini_test)                        # 生成 HTML 报告            report_path = "docs/learning_journey/eval_demo_report.html"            evaluator.generate_html_report(results, report_path)                        print(f"\n📊 评估结果:")            print(f"   BLEU-4 均分: {results['bleu_avg']:.4f}")            print(f"   ROUGE-L F1: {results['rouge_l_f1_avg']:.4f}")            print(f"   教育质量总分: {results['education_quality_overall']:.1f}/10")            print(f"   费曼度: {results['feynman_score_avg']:.1f}%")            print(f"   平均延迟: {results['latency_avg_ms']:.0f}ms")            print(f"\n📄 HTML 报告: {os.path.abspath(report_path)}")        else:            print("⚠️ 未找到 qwen 模型")    else:        print(f"⚠️ Ollama 不可用 (状态码: {resp.status_code})")except requests.exceptions.ConnectionError:    print("⚠️ 无法连接到 Ollama，跳过模型评估")    print("\n   💡 手动评估命令:")    print("   python -m models.distil.evaluate --model qwen2.5:7b --output eval_report.html")except Exception as e:    print(f"⚠️ 评估失败: {e}")    print("\n   💡 这可能是因为缺少依赖或 Ollama 未运行")

---

## 第 7 章：GGUF 转换与 Ollama 部署

### 为什么需要 GGUF？

训练完成后，模型权重是 HuggingFace 格式（.safetensors / .bin）。
要让 Ollama 加载你的模型，需要：

1. **合并 LoRA adapter** → 将 adapter 权重合并到基座模型
2. **转换为 GGUF** → 使用 llama.cpp 的 convert 工具
3. **量化** → 减少模型大小（q4_k_m, q5_k_m 等）
4. **创建 Modelfile** → 定义系统提示词和参数
5. **ollama create** → 注册到 Ollama

下面我们来看看 Modelfile 的结构：

In [ ]:
# ==================== 第7章：Modelfile 模板 ====================print("=" * 60)print("📋 Ollama Modelfile 模板")print("=" * 60)modelfile_content = '''# LumiLearn 自研模型 - lumilearn-teacher-v1# 生成时间: 2026-06-03 15:00:00# 基座模型: qwen2.5:7bFROM qwen2.5:7b# 系统提示词SYSTEM """你是一位专业的AI教师，精通费曼五步教学法。你的教学风格：1. 用简单的类比和比喻解释复杂概念2. 逐步引导，从基础到深入3. 鼓励提问和思考4. 使用具体例子辅助理解5. 总结关键要点，强化记忆请根据学生的提问给出详细、准确、易懂的回答。对于学科问题，使用费曼五步法：- Step 1: 确定教学目标- Step 2: 用简单语言解释- Step 3: 找出理解盲区- Step 4: 回顾并简化- Step 5: 总结和知识迁移"""# 模型参数PARAMETER temperature 0.7PARAMETER top_p 0.9PARAMETER top_k 40PARAMETER num_predict 2048'''print(modelfile_content)print("=" * 60)print("📋 创建 Ollama 模型的命令:")print("=" * 60)print("  ollama create lumilearn-teacher-v1 -f Modelfile")print("  ollama run lumilearn-teacher-v1")print()print("💡 如果使用本地 GGUF 文件:")print("  FROM ./models/distil/output/gguf/lumilearn_model_q4_k_m.gguf")

In [ ]:
# ==================== 第7章：GGUF 导出模拟 ====================print("\n" + "=" * 60)print("🔧 GGUF 导出流程模拟")print("=" * 60)from models.distil.trainer import DistilTrainerprint("\n📦 GGUF 导出流程:")print("   Step 1: 加载基座模型 + LoRA adapter")print("   Step 2: 合并 adapter 权重到基座模型")print("   Step 3: 保存合并后的完整模型")print("   Step 4: 调用 llama.cpp convert.py 转换为 GGUF")print("   Step 5: 使用 quantize 工具量化")# 模拟导出print("\n💡 代码示例:")print('''from models.distil.trainer import DistilTrainertrainer = DistilTrainer(model_name="Qwen/Qwen2.5-7B-Instruct")trainer.load_model(use_4bit=False)gguf_path = trainer.export_gguf(    adapter_path="models/distil/adapter",    output_path="models/distil/output/gguf",    quantize="q4_k_m")print(f"GGUF 文件: {gguf_path}")''')print("\n⚠️ 注意:")print("   - GGUF 转换需要 llama.cpp 工具")print("   - 安装: git clone https://github.com/ggerganov/llama.cpp && cd llama.cpp && make")print("   - 量化方法: q4_0 (最快), q4_k_m (推荐), q5_k_m (更高质量), q8_0 (最高质量)")print("   - 7B 模型量化后大小: q4_k_m ≈ 4.5GB, q8_0 ≈ 7.5GB")

---

## 第 8 章：框架集成 —— 将模型注册到 LumiLearn

### 模型如何注册到框架？

LumiLearn 使用 `VersionManager` 管理模型版本：

- **注册**：将训练好的模型信息记录到 `versions.json`
- **切换**：在不同版本之间切换（包括回退到基座模型）
- **回滚**：一键回滚到上一个版本
- **对比**：两个版本的评估指标对比

下面我们来实际操作 VersionManager：

In [ ]:
# ==================== 第8章：VersionManager 版本管理 ====================from models.version_manager import VersionManagerprint("=" * 60)print("📋 模型版本管理器")print("=" * 60)# 初始化 VersionManagervm = VersionManager()# 列出现有版本versions = vm.list_versions()active = vm.get_active()if versions:    print(f"\n当前激活版本: {active}")    print(f"已注册版本数: {len(versions)}")    for v in versions:        status = "🟢 激活" if v.get("status") == "active" else "⚪ 非激活"        print(f"\n  {status} {v['name']}")        print(f"    基座模型: {v.get('base_model', 'N/A')}")        print(f"    创建时间: {v.get('created_at', 'N/A')}")        print(f"    训练耗时: {v.get('training_time', 'N/A')}")else:    print("\n📭 暂无已注册的模型版本")    print("   训练完成后，管线会自动注册版本")    print()    print("   💡 手动注册命令:")    print("   python -m models.version_manager register \\")    print("       --name lumilearn-teacher-v1 \\")    print("       --base-model qwen2.5:7b \\")    print("       --gguf-path models/distil/output/gguf/lumilearn_model_q4_k_m.gguf \\")    print("       --eval-file eval_report.json \\")    print("       --training-time 2h15m \\")    print("       --sample-count 450 \\")    print("       --description '首次QLoRA微调，费曼五步法教学风格'")# 显示切换历史history = vm.get_version_history()if history:    print(f"\n📜 版本切换历史:")    for i, h in enumerate(history, 1):        marker = " ← 当前" if h == active else ""        print(f"  {i}. {h}{marker}")

In [ ]:
# ==================== 第8章：演示注册一个版本 ====================print("\n" + "=" * 60)print("🧪 演示：注册一个模拟版本")print("=" * 60)# 注册一个演示版本demo_version = vm.register_version(    name="demo-teacher-v1",    base_model="qwen2.5:7b",    gguf_path="models/distil/output/gguf/demo_model_q4_k_m.gguf",    eval_scores={        "bleu": 0.32,        "rouge_l": 0.58,        "education_quality": 7.8,        "feynman_style": 72.5    },    training_time="1h30m",    description="演示版本 - 费曼五步法教学风格",    sample_count=100)print("\n注册信息:")print(f"  名称: {demo_version['name']}")print(f"  基座: {demo_version['base_model']}")print(f"  状态: {demo_version['status']}")print(f"  评估分数: {demo_version['eval_scores']}")# 现在列出所有版本print("\n更新后的版本列表:")for v in vm.list_versions():    status = "🟢" if v.get("status") == "active" else "⚪"    print(f"  {status} {v['name']} ({v.get('base_model', 'N/A')})")

In [ ]:
# ==================== 第8章：测试模型 API ====================print("\n" + "=" * 60)print("🌐 测试模型 API 端点")print("=" * 60)try:    import requests        # 测试 LumiLearn API    api_url = "http://localhost:18080/api/chat"    print(f"\n测试端点: {api_url}")        try:        resp = requests.post(            api_url,            json={                "message": "请用一句话解释什么是勾股定理",                "model": "qwen2.5:7b"            },            timeout=10        )        if resp.status_code == 200:            data = resp.json()            print(f"✅ API 响应: {data.get("response", str(data))[:200]}...")        else:            print(f"⚠️ HTTP {resp.status_code}: {resp.text[:200]}")    except requests.exceptions.ConnectionError:        print("⚠️ LumiLearn API 未运行 (localhost:18080)")        print("   启动命令: python app.py")    except Exception as e:        print(f"⚠️ 请求失败: {e}")        # 测试 Ollama API    ollama_url = "http://192.168.2.xx:11434/api/chat"    print(f"\n测试端点: {ollama_url}")        try:        resp = requests.post(            ollama_url,            json={                "model": "qwen2.5:7b",                "messages": [{"role": "user", "content": "1+1=?"}],                "stream": False            },            timeout=10        )        if resp.status_code == 200:            data = resp.json()            content = data.get("message", {}).get("content", "")            print(f"✅ Ollama 响应: {content[:200]}")        else:            print(f"⚠️ HTTP {resp.status_code}")    except requests.exceptions.ConnectionError:        print("⚠️ Ollama 未运行 (192.168.2.xx:11434)")    except Exception as e:        print(f"⚠️ 请求失败: {e}")except ImportError:    print("⚠️ requests 库未安装")    print("   安装: pip install requests")except Exception as e:    print(f"⚠️ 发生错误: {e}")

---

## 第 9 章：A/B 对比 —— 你的模型 vs 基座模型

### 为什么要做 A/B 对比？

模型对比能帮你量化微调的效果：
- 微调后 BLEU 提升了多少？
- 教育质量有没有改善？
- 费曼风格是否更明显？
- 延迟有没有增加？

下面我们用 `ModelEvaluator.compare_models()` 来对比两个模型：

In [ ]:
# ==================== 第9章：A/B 模型对比 ====================print("=" * 60)print("🔬 A/B 模型对比")print("=" * 60)try:    import requests    from models.distil.evaluate import ModelEvaluator, DEFAULT_TEST_SET        # 检查 Ollama    resp = requests.get("http://192.168.2.xx:11434/api/tags", timeout=5)    if resp.status_code == 200:        model_names = [m["name"] for m in resp.json().get("models", [])]        qwen_models = [m for m in model_names if "qwen" in m.lower()]                if len(qwen_models) >= 2:            model_a, model_b = qwen_models[0], qwen_models[1]            print(f"\n对比模型: {model_a} vs {model_b}")            print("⏳ 正在运行对比评估（仅前2题）...")                        evaluator = ModelEvaluator()            mini_test = DEFAULT_TEST_SET[:2]            comparison = evaluator.compare_models(model_a, model_b, test_set=mini_test)                        # 生成对比报告            report_path = "docs/learning_journey/compare_demo_report.html"            evaluator.generate_html_report(comparison, report_path)                        # 显示对比结果            comp = comparison["comparison"]            print(f"\n📊 对比结果:")            print(f"   BLEU:     {comp['bleu']['model_a']:.4f} vs {comp['bleu']['model_b']:.4f} → {comp['bleu']['winner']}")            print(f"   ROUGE-L:  {comp['rouge_l']['model_a']:.4f} vs {comp['rouge_l']['model_b']:.4f} → {comp['rouge_l']['winner']}")            print(f"   费曼度:   {comp['feynman']['model_a']:.1f}% vs {comp['feynman']['model_b']:.1f}% → {comp['feynman']['winner']}")            print(f"   综合胜者: {comp['overall']['winner']}")            print(f"\n📄 对比报告: {os.path.abspath(report_path)}")                    elif len(qwen_models) == 1:            print(f"⚠️ 只有 1 个 qwen 模型 ({qwen_models[0]})，无法对比")            print("   需要至少 2 个模型: ollama pull qwen2.5:14b")        else:            print("⚠️ 未找到 qwen 模型")    else:        print("⚠️ Ollama 不可用")except requests.exceptions.ConnectionError:    print("⚠️ 无法连接 Ollama，跳过 A/B 对比")except Exception as e:    print(f"⚠️ 对比失败: {e}")        # 模拟对比结果    print("\n💡 模拟对比结果 (base vs fine-tuned):")    print("   " + "=" * 50)    print(f"   {'指标':<16} {'base':<10} {'fine-tuned':<12} {'差异':>8}   胜者")    print("   " + "-" * 50)    print(f"   {'BLEU-4':<16} {0.18:<10.4f} {0.32:<12.4f} {+0.14:>8.4f}   fine-tuned")    print(f"   {'ROUGE-L':<16} {0.35:<10.4f} {0.58:<12.4f} {+0.23:>8.4f}   fine-tuned")    print(f"   {'费曼度':<16} {35.0:<10.1f}% {72.5:<11.1f}% {+37.5:>8.1f}%  fine-tuned")    print(f"   {'教育质量':<16} {5.2:<10.1f} {7.8:<11.1f} {+2.6:>8.1f}   fine-tuned")    print(f"   {'延迟':<16} {850:<10.0f}ms {920:<11.0f}ms {+70:>8.0f}ms  base")    print("   " + "=" * 50)    print("   综合胜者: fine-tuned (4:1)")

In [ ]:
# ==================== 第9章：雷达图可视化 ====================import matplotlib.pyplot as pltimport numpy as npimport mathprint("\n" + "=" * 60)print("📊 雷达图：Base Model vs Fine-tuned Model")print("=" * 60)# 模拟数据categories = ['正确性', '完整性', '难度适应', '引导性', '趣味性']base_scores = [6.5, 5.0, 4.0, 3.5, 3.0]finetuned_scores = [8.5, 7.5, 8.0, 7.0, 8.0]N = len(categories)angles = [n / float(N) * 2 * math.pi for n in range(N)]angles += angles[:1]  # 闭合base_scores += base_scores[:1]finetuned_scores += finetuned_scores[:1]# 创建雷达图fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))ax.fill(angles, base_scores, alpha=0.25, color='#667eea', label='Base Model (qwen2.5:7b)')ax.plot(angles, base_scores, color='#667eea', linewidth=2)ax.fill(angles, finetuned_scores, alpha=0.25, color='#f5576c', label='Fine-tuned (lumilearn-teacher)')ax.plot(angles, finetuned_scores, color='#f5576c', linewidth=2)ax.set_xticks(angles[:-1])ax.set_xticklabels(categories, fontsize=12)ax.set_ylim(0, 10)ax.set_yticks([2, 4, 6, 8, 10])ax.set_yticklabels(['2', '4', '6', '8', '10'], fontsize=8)ax.set_title('教育质量五维度对比', fontsize=16, fontweight='bold', pad=20)ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("\n📊 改进幅度:")for cat, base, ft in zip(categories, base_scores[:-1], finetuned_scores[:-1]):    improvement = ft - base    bar = "+" * int(improvement * 2) if improvement > 0 else "-" * int(abs(improvement) * 2)    print(f"  {cat:<8}: {base:.1f} → {ft:.1f} ({improvement:+.1f}) {bar}")print(f"\n  平均提升: {np.mean(np.array(finetuned_scores[:-1]) - np.array(base_scores[:-1])):+.1f} 分")

---

## 第 10 章：练习与扩展

恭喜你完成了完整的训练流程学习！下面是 5 个实战练习，帮助你巩固知识：

### 练习 1：添加新学科模板

在 `models/distil/prompts.py` 中添加一个生物学（biology）的 system prompt 和至少 5 个 topic 模板。

### 练习 2：调整训练超参数

修改 `lora_r`（8/16/32）、`learning_rate`（1e-4/2e-4/5e-4）、`epochs`（1/3/5），观察 loss 曲线的变化。

### 练习 3：自定义测试集评估

创建你自己的测试集（JSON 格式），运行 `ModelEvaluator.evaluate_model()` 进行评估。

### 练习 4：更换基座模型

尝试使用 `qwen2.5:14b` 或 `llama3.1:8b` 作为基座模型进行训练。

### 练习 5：部署模型并构建 Web UI

将训练好的模型部署到 LumiLearn 框架，并构建一个简单的 Web 界面来测试。

下面是一些练习的代码框架：

In [ ]:
# ==================== 第10章：练习1 - 添加新学科模板 ====================print("=" * 60)print("📝 练习 1: 添加生物学学科模板")print("=" * 60)print("""在 models/distil/prompts.py 中添加以下内容：# 生物 System PromptBIOLOGY_SYSTEM_PROMPT = \"\"\"你是一位生物教师，擅长用进化论视角解释生命现象。...（参考数学/物理的 system prompt 格式）\"\"\"# 生物 Topic 模板BIOLOGY_TOPICS = [    {"subject": "biology", "topic": "细胞结构", "prompt_template": "..."},    {"subject": "biology", "topic": "DNA复制", "prompt_template": "..."},    {"subject": "biology", "topic": "光合作用", "prompt_template": "..."},    {"subject": "biology", "topic": "自然选择", "prompt_template": "..."},    {"subject": "biology", "topic": "生态系统", "prompt_template": "..."},]然后注册到 get_subject_training_templates() 函数中。""")print("💡 提示: 参考现有学科（math/physics/chemistry）的模板格式")

In [ ]:
# ==================== 第10章：练习2 - 超参数对比 ====================import numpy as npimport matplotlib.pyplot as pltprint("\n" + "=" * 60)print("📝 练习 2: 超参数对 Loss 的影响")print("=" * 60)np.random.seed(42)steps = np.arange(0, 500)# 不同学习率的影响fig, axes = plt.subplots(1, 3, figsize=(16, 5))# 学习率对比for lr, color, label in [(1e-4, '#43e97b', 'lr=1e-4'),                           (2e-4, '#667eea', 'lr=2e-4 (默认)'),                           (5e-4, '#f5576c', 'lr=5e-4')]:    loss = 3.0 * np.exp(-steps / (150 * (2e-4/lr))) + 0.3    loss += np.random.normal(0, 0.05 * (lr/2e-4), len(steps))    axes[0].plot(steps, loss, color=color, linewidth=1.5, alpha=0.7, label=label)axes[0].set_title('学习率对 Loss 的影响', fontsize=12, fontweight='bold')axes[0].set_xlabel('步数')axes[0].set_ylabel('Loss')axes[0].legend()axes[0].grid(True, alpha=0.3)# LoRA rank 对比for r, color, label in [(8, '#f093fb', 'r=8'),                          (16, '#667eea', 'r=16 (默认)'),                          (32, '#4facfe', 'r=32')]:    loss = 3.0 * np.exp(-steps / 150) + 0.35 - 0.02 * (r/8)    loss += np.random.normal(0, 0.04, len(steps))    axes[1].plot(steps, loss, color=color, linewidth=1.5, alpha=0.7, label=label)axes[1].set_title('LoRA Rank 对 Loss 的影响', fontsize=12, fontweight='bold')axes[1].set_xlabel('步数')axes[1].legend()axes[1].grid(True, alpha=0.3)# Epochs 对比for ep, color, label in [(1, '#fa709a', '1 epoch'),                           (3, '#667eea', '3 epochs (默认)'),                           (5, '#43e97b', '5 epochs')]:    loss = 3.0 * np.exp(-steps / (500/ep)) + 0.4 - 0.05 * ep    loss += np.random.normal(0, 0.04, len(steps))    axes[2].plot(steps, loss, color=color, linewidth=1.5, alpha=0.7, label=label)axes[2].set_title('训练轮数对 Loss 的影响', fontsize=12, fontweight='bold')axes[2].set_xlabel('步数')axes[2].legend()axes[2].grid(True, alpha=0.3)plt.suptitle('超参数对训练的影响', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()print("\n💡 实践建议:")print("   - 学习率: 从 2e-4 开始，如果 loss 震荡则降低")print("   - LoRA rank: r=16 是 sweet spot，数据多可尝试 r=32")print("   - Epochs: 3 轮通常足够，数据少时增加但注意过拟合")

In [ ]:
# ==================== 第10章：练习3 - 自定义测试集评估 ====================print("\n" + "=" * 60)print("📝 练习 3: 创建自定义测试集")print("=" * 60)custom_test_set = [    {        "subject": "biology",        "topic": "细胞呼吸",        "question": "请用费曼教学法讲解细胞呼吸的过程（糖酵解、三羧酸循环、氧化磷酸化）。",        "reference_answer": "细胞呼吸是将葡萄糖转化为ATP的过程。糖酵解在细胞质中分解葡萄糖为丙酮酸，产生2个ATP。三羧酸循环在线粒体基质中完成，氧化磷酸化在线粒体内膜进行，产生最多ATP（约34个）。"    },    {        "subject": "biology",        "topic": "遗传学",        "question": "请讲解孟德尔遗传定律中的分离定律和自由组合定律。",        "reference_answer": "分离定律：等位基因在形成配子时分离，每个配子只含一个等位基因。自由组合定律：不同基因的等位基因在配子形成时独立分配。例如AaBb个体产生AB、Ab、aB、ab四种配子，比例1:1:1:1。"    },]print("\n自定义测试集内容:")for i, item in enumerate(custom_test_set):    print(f"\n  [{i+1}] {item['subject']}/{item['topic']}")    print(f"  问题: {item['question'][:80]}...")print("\n💡 使用方式:")print("from models.distil.evaluate import ModelEvaluator")print("evaluator = ModelEvaluator()")print("results = evaluator.evaluate_model('qwen2.5:7b', test_set=custom_test_set)")print("evaluator.generate_html_report(results, 'custom_eval.html')")

In [ ]:
# ==================== 第10章：练习4 - 更换基座模型 ====================print("\n" + "=" * 60)print("📝 练习 4: 更换基座模型")print("=" * 60)print("""支持的基座模型:  模型                 参数量    显存需求    推荐场景  ─────────────────────────────────────────────────  qwen2.5:7b          7B        ~6GB VRAM   默认推荐  qwen2.5:14b         14B       ~12GB VRAM  更高质量  qwen2.5:3b          3B        ~3GB VRAM   快速实验  llama3.1:8b         8B        ~7GB VRAM   替代选择  mistral:7b          7B        ~6GB VRAM   推理速度快使用方式:  # 下载模型  ollama pull qwen2.5:14b    # 修改训练命令  bash train_lumilearn.sh --base qwen2.5:14b --hf-model Qwen/Qwen2.5-14B-Instruct    # 或在代码中  from models.distil.trainer import DistilTrainer  trainer = DistilTrainer(model_name="Qwen/Qwen2.5-14B-Instruct")""")print("💡 注意: 14B 模型需要更多显存和磁盘空间")

In [ ]:
# ==================== 第10章：练习5 - 构建简单 Web UI ====================print("\n" + "=" * 60)print("📝 练习 5: 部署模型并构建 Web UI")print("=" * 60)print("""完整部署流程:1. 训练模型   bash train_lumilearn.sh --model-name my-teacher-v12. 测试模型   ollama run my-teacher-v1   >>> 请用费曼五步法讲解勾股定理3. 构建 Web UI (使用 Flask)   from flask import Flask, request, jsonify, render_template_string   import requests      app = Flask(__name__)      HTML_TEMPLATE = '''   <!DOCTYPE html>   <html>   <head><title>LumiLearn Teacher</title></head>   <body>       <h1>🐝 LumiLearn 教师模型</h1>       <textarea id="question" rows="3" cols="60"></textarea>       <button onclick="ask()">提问</button>       <div id="answer"></div>       <script>       async function ask() {           const q = document.getElementById('question').value;           const resp = await fetch('/api/chat', {               method: 'POST',               headers: {'Content-Type': 'application/json'},               body: JSON.stringify({message: q})           });           const data = await resp.json();           document.getElementById('answer').innerHTML = data.response;       }       </script>   </body>   </html>   '''      @app.route('/')   def index():       return render_template_string(HTML_TEMPLATE)      @app.route('/api/chat', methods=['POST'])   def chat():       msg = request.json.get('message', '')       resp = requests.post('http://192.168.2.xx:11434/api/chat', json={           'model': 'my-teacher-v1',           'messages': [{'role': 'user', 'content': msg}],           'stream': False       })       return jsonify({'response': resp.json()['message']['content']})      if __name__ == '__main__':       app.run(host='0.0.0.0', port=5000)""")print("💡 访问 http://localhost:5000 即可使用你的自定义教师模型！")

---

🎉 总结

恭喜你完成了完整的 LumiLearn 模型训练教程！

### 你学到了什么

| 技能 | 掌握程度 |
|------|----------|
| 理解 QLoRA 原理（量化 + 低秩适应） | ✅ |
| 使用 DataFactory 生成训练数据 | ✅ |
| 运行 7 步训练管线 | ✅ |
| 分析损失曲线，诊断训练问题 | ✅ |
| 使用 BLEU/ROUGE/教育质量评估模型 | ✅ |
| GGUF 转换与 Ollama 部署 | ✅ |
| VersionManager 版本管理 | ✅ |
| A/B 模型对比 | ✅ |

### 下一步

1. **实际训练**: 运行 `bash train_lumilearn.sh` 训练你自己的模型
2. **数据扩展**: 添加更多学科模板，提高数据质量
3. **超参数调优**: 尝试不同的 lora_r、learning_rate、epochs
4. **模型对比**: 对比不同基座模型的微调效果
5. **部署上线**: 将模型集成到 LumiLearn 框架中

### 相关资源

- 项目文档: `docs/ARCHITECTURE.md`
- 学习路径: `docs/learning_journey/INDEX.md`
- 评价模块: `docs/learning_journey/Module_4.3_Prompt工程.md`
- 部署指南: `docs/learning_journey/deploy_guide.ipynb`

---

> 🐝 **记住**：训练模型不是终点，而是起点。持续迭代数据、优化超参数、对比评估，才能打造出真正优秀的 AI 教师！